# Conexión a Supabase PostgreSQL y listado de tablas

Este notebook se conecta a una base de datos PostgreSQL de Supabase y lista las tablas visibles para el usuario.

> Seguridad: no escribas ni publiques tu contraseña dentro del notebook. La conexión solicita la contraseña de forma oculta.

## 1. Instalar dependencias

Ejecuta esta celda una sola vez por entorno.

In [1]:
%pip install -q sqlalchemy psycopg2-binary pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Definir los parámetros de conexión

Obtén `host`, `port`, `database` y `username` desde **Supabase Dashboard → Connect**. Para DBeaver y scripts persistentes normalmente se usa el **Session Pooler**.

Ejemplos habituales:
- `host`: `aws-0-tu-region.pooler.supabase.com`
- `port`: `5432`
- `database`: `postgres`
- `username`: `postgres.tu_project_ref`

No modifiques la variable `password` en el código: se solicitará de forma segura al ejecutar la celda.

In [2]:
import os
from getpass import getpass
from urllib.parse import quote_plus

HOST = "aws-0-us-west-2.pooler.supabase.com"  # Dato generado por Supabase
PORT = 5432
DATABASE = "postgres"
USERNAME = "postgres.prirpipdfdcycvdwsyvn"  # USuario de Supabase

password = os.getenv("SUPABASE_DB_PASSWORD")
if not password:
    password = getpass("Contraseña de la base de datos Supabase: ")

print("Parámetros cargados. La contraseña no se mostrará ni se guardará en el notebook.")

Parámetros cargados. La contraseña no se mostrará ni se guardará en el notebook.


## 3. Crear y probar la conexión

La opción `sslmode=require` cifra el tráfico entre el notebook y Supabase.

In [3]:
from sqlalchemy import create_engine, text

connection_url = (
    f"postgresql+psycopg2://{quote_plus(USERNAME)}:{quote_plus(password)}"
    f"@{HOST}:{PORT}/{DATABASE}?sslmode=require"
)

engine = create_engine(connection_url, pool_pre_ping=True)

with engine.connect() as connection:
    version = connection.execute(text("SELECT version();")).scalar()

print("Conexión exitosa.")
print(version)

Conexión exitosa.
PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


## 4. Listar schemas y tablas

Esta consulta excluye schemas internos de PostgreSQL y muestra las tablas base a las que tu usuario puede acceder.

In [6]:
import pandas as pd
from sqlalchemy import text

query_tablas = text("""
SELECT
    table_schema AS schema,
    table_name AS tabla,
    table_type AS tipo
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
  AND table_schema NOT LIKE 'pg_toast%'
ORDER BY table_schema, table_name;
""")

#tablas_df = pd.read_sql(query_tablas, engine)
with engine.connect() as connection:
    tablas_df = pd.read_sql_query(query_tablas, connection)

print(f"Tablas encontradas: {len(tablas_df)}")
display(tablas_df)

Tablas encontradas: 57


,schema,tabla,tipo
0,auth,audit_log_entries,BASE TABLE
1,auth,custom_oauth_providers,BASE TABLE
2,auth,flow_state,BASE TABLE
3,auth,identities,BASE TABLE
4,auth,instances,BASE TABLE
5,auth,mfa_amr_claims,BASE TABLE
6,auth,mfa_challenges,BASE TABLE
7,auth,mfa_factors,BASE TABLE
8,auth,mfa_recovery_code_sets,BASE TABLE
9,auth,mfa_recovery_codes,BASE TABLE


## 5. Filtrar un schema específico (opcional)

Cambia `SCHEMA_OBJETIVO` por el schema que deseas explorar, por ejemplo `public`, `recaudo`, `operacion` o `analytics`.

In [10]:
from sqlalchemy import text

schema_objetivo = "bronze"

query_schema = text("""
SELECT
    table_name AS tabla,
    table_type AS tipo
FROM information_schema.tables
WHERE table_schema = :schema_objetivo
ORDER BY table_name;
""")

with engine.connect() as connection:
    tablas_schema_df = pd.read_sql_query(
        query_schema,
        connection,
        params={"schema_objetivo": schema_objetivo}
    )

display(tablas_schema_df)

,tabla,tipo
0,dia_tipo,BASE TABLE
1,usos,BASE TABLE
2,usos_validador,BASE TABLE


## 6. Ver las columnas de una tabla (opcional)

Define el schema y la tabla. Esta celda muestra nombre de columna, tipo, nulabilidad y valor por defecto.

In [11]:
SCHEMA_TABLA = "bronze"
TABLA_OBJETIVO = "dia_tipo"  # Reemplaza este valor

query_columnas = text("""
SELECT
    ordinal_position AS posicion,
    column_name AS columna,
    data_type AS tipo_dato,
    is_nullable AS permite_nulos,
    column_default AS valor_por_defecto
FROM information_schema.columns
WHERE table_schema = :schema
  AND table_name = :tabla
ORDER BY ordinal_position;
""")

with engine.connect() as connection:
    columnas_df = pd.read_sql(
        query_columnas,
        connection,
        params={"schema": SCHEMA_TABLA, "tabla": TABLA_OBJETIVO}
    )

display(columnas_df)

,posicion,columna,tipo_dato,permite_nulos,valor_por_defecto
0,1,id,bigint,NO,None
1,2,archivo_id,uuid,NO,None
2,3,fila_origen,integer,NO,None
3,4,fecha,date,NO,None
4,5,dia_tipo,text,NO,None
5,6,archivo_origen,text,NO,None
6,7,cargado_en,timestamp with time zone,NO,now()


## 7. Cerrar conexiones

Ejecuta esta celda al finalizar el trabajo.

In [12]:
engine.dispose()
print("Pool de conexiones cerrado.")

Pool de conexiones cerrado.
